# COMET InSAR Workshop: LiCSBAS Tutorial — **SBOVNL (SBOI) data processing** (`--sbovl` chain)*SBOVNL (burst-overlap Very short (baseline) Network) is an **alternative data product and referencing scheme** based on S-1 *burst* overlaps. It runs as a *separate chain*: download (s)bovl data with `--sbovl`, multilook it, then run the time series steps `11` → `16` **all with `--sbovl`** (the referencing in `120` is forced on, and classic loop closure `12` is skipped).This notebook covers the **whole** SBOVNL chain (steps `0-1` → `1-6`) because *both* modules differ from the standard route. For the standard LiCSAR route see [Module 0](tutorial_0.ipynb) + [Module 1](tutorial_1.ipynb).

### please run the following cell first to load some necessary modules:  
(to run a cell, just click into it and press shift-Enter)  

#### Note: in case your notebook gets frozen/kernel disconnected, try clicking Kernel->Restart (rather than Refreshing page). If this runs ok, you may just rerun the following cell, and then skip back to the cell you were in before.

In [ ]:
import sys, os, subprocess, warnings
import importlib.util as _iu
from IPython.display import Image

def _run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.returncode != 0:
        print((r.stdout or "")[-3000:])
        raise RuntimeError("command failed: " + cmd)

def _have(mod):
    try:
        return _iu.find_spec(mod) is not None
    except Exception:
        return False

# COMET local jupyterhub: add the legacy bash CLI wrappers (bin/) to PATH, if present
_hub = '/data/notebooks/repositories/LiCSBAS/bin'
if os.path.exists(_hub):
    os.environ['PATH'] = os.environ['PATH'] + ':' + _hub

# ---- LiCSBAS v3 is a pip package ('licsbas'). Make sure the (binary) GDAL stack
# ---- and the package itself are available before anything imports them.
# ---- This is idempotent: on a kernel that already has them, it is a no-op.
_installed = []

# 1) GDAL Python bindings (osgeo) + pyproj/rasterio are the hard binary deps of
#    licsbas. 'osgeo' is NOT on PyPI, so on an environment that has conda
#    (Colab, Binder) we pull it from conda-forge; otherwise it must already be
#    present in the system/conda env.
if not _have("osgeo"):
    if os.path.exists("/opt/conda/bin/conda") or _have("conda"):
        print("GDAL (osgeo) not found - installing from conda-forge (this can take a minute) ...")
        _run("conda install -y -q -c conda-forge gdal pyproj=3.6")
        _installed.append("gdal (conda-forge)")
    else:
        raise SystemExit(
            "GDAL Python bindings ('osgeo') are required by licsbas but not found.\n"
            "Install them, e.g., with:\n"
            "    conda install -c conda-forge gdal pyproj=3.6\n"
            "(on Google Colab this cell should auto-install them via conda-forge).")

# 2) the licsbas package itself
if not _have("licsbas"):
    print("licsbas not found - installing 'licsbas[viewer]' via pip ...")
    _run('pip install -q "licsbas[viewer]"')
    _installed.append("licsbas[viewer]")

warnings.filterwarnings('ignore')

if _installed:
    print("\n[setup] installed: " + ", ".join(_installed))
    print("[setup] New native libraries need a KERNEL RESTART to be picked up.")
    print("[setup]   -> Runtime > Restart session (or Restart kernel), then re-run THIS cell.\n")
    raise SystemExit("Please restart the kernel and re-run this cell to finish setup.")

# some extra functions
def display(png):
    if not os.path.exists(png):
        print('This png file does not exist, recheck please')
    else:
        return Image(png)

def printbold(instr='done'):
    print('\033[1m' + instr + '\033[0m')

# selected frame
frame = '022D_04826_121209'  # for tutorial

# locate the pre-extracted tutorial dataset (binder postBuild / install script)
framedir = None
for cand in (os.getcwd(),
             os.path.join(os.getcwd(), frame),
             os.path.join(os.environ.get('HOME', '.'), frame)):
    if os.path.exists(os.path.join(cand, 'GEOCml2GACOS')):
        framedir = cand
        break

if framedir is None:
    print('WARNING: tutorial dataset not found (GEOCml2GACOS).')
    print('Download and extract it first - see tutorial/install_complete.sh or the binder postBuild script.')
else:
    os.chdir(framedir)

print('prerequisites loaded, thank you. Running in directory:')
print(os.getcwd())
printbold()


## The SBOVNL processing chain (steps `0-1` → `1-6`, all with `--sbovl`)<img src="https://raw.githubusercontent.com/comet-licsar/LiCSBAS/towards-v3/tutorial/0_sbovl.png" width="65%">*(image not visible? the PNGs also live in `tutorial/0_sbovl.png`; `python3 tutorial/make_flowchart.py` regenerates them)*

## Step 0-1: **download SBOVNL data** — `LiCSBAS01_get_geotiff.py --sbovl`Same download step as the standard route, but with `--sbovl` it pulls the **(s)bovl** products from the portal instead of the unwrapped data (and, optionally, the *azimuth iono/tides* that `131` will use):```[- yyyymmdd_yyyymmdd.geo.s(bovldiff).adf.mm.tif] (if --sbovl is used)[- yyyymmdd_yyyymmdd.geo.s(bovldiff).adf.cc.tif] (if --sbovl is used)```

In [ ]:
![-h help] licsbas-01-get-geotiff -h

In [ ]:
# EXAMPLE (SBOVNL: note --sbovl, and the azi iono/tides if you want step 131)# licsbas-01-get-geotiff -f 022D_04826_121209 --sbovl --get_iono --get_tides

## Step 0-2: **multilook the SBOVNL data** — `LiCSBAS02_ml_prep.py --sbovl`Multilooking on the (s)bovl `.mm`/`.cc` pairs. Convention: put the SBOVNL output in a **separate** dir, e.g. `GEOC_BOVL_ml5` (it must *not* overwrite the standard `GEOCml5`):```LiCSBAS02_ml_prep.py -i GEOCdir -o GEOC_BOVL_ml5 -n 5 --sbovl```

In [ ]:
![-h help] licsbas-02-ml-prep -h# EXAMPLE# licsbas-02-ml-prep -i GEOCdir -o GEOC_BOVL_ml5 -n 5 --sbovl

## Optional steps `0-3` → `0-5` (GACOS / mask / clip)The optional `GACOS` correction, the coherence mask and the clipping are **exactly as in the standard [Module-0 tutorial](tutorial_0.ipynb)** — they are not part of the SBOVNL scheme itself, and are typically *skipped* for (s)bovl (the scheme is already referenced on very short baselines).

## Step 1-1: **quality check** — `LiCSBAS11_check_unw.py --sbovl`Checks the (s)bovl stack for coherence/unwrapping outliers and prepares the TS dir. With `--sbovl` this step **forces the reference selection (step `120`) on**, because (s)bovl data are already referenced on the *overlap* (no classic re-referencing on a stable point).> In the SBOVNL chain, step `12` (loop closure) is **skipped** — the re-referencing is done on the very-short-overlap network instead.

In [ ]:
![-h help] licsbas-11-check-unw -h# EXAMPLE# licsbas-11-check-unw -d GEOC_BOVL_ml5 -t TS_GEOC_BOVL_ml5 --sbovl -c 0.05 -u 0.3

## Step 1-2: **choose the (s)bovl reference point** — `LiCSBAS120_choose_reference.py --sbovl`Used (and forced) in the SBOVNL chain to pick/reference a *stable point* within the (s)bovl network (it builds the TS reference on the very-short baselines):```LiCSBAS120_choose_reference.py -c GEOC_BOVL_ml5 -d GEOC_BOVL_ml5 -t TS_... --sbovl```

In [ ]:
![-h help] licsbas-120-choose-reference -h# EXAMPLE# licsbas-120-choose-reference -c GEOC_BOVL_ml5 -d GEOC_BOVL_ml5 -t TS_GEOC_BOVL_ml5 --sbovl

## Step 1-3: **SBOVNL inversion** — `LiCSBAS13_sb_inv.py --sbovl` (or `--sbovl_abs`)Two flavours:- `--sbovl` — inversion of the (s)bovl `mm` + `cc` values (the network is already close-referenced; *no re-referencing is done here*).- `--sbovl_abs` — **SBOI absolute**: recompute the *absolute* velocity (the reference/correction steps are skipped); afterwards `LiCSBAS131_boi_absolute.py` can apply the **ICC + ESD** offset and the absolute SET/Iono corrections.```LiCSBAS13_sb_inv.py -d GEOC_BOVL_ml5 -t TS_GEOC_BOVL_ml5 --sbovlLiCSBAS13_sb_inv.py -d GEOC_BOVL_ml5 -t TS_GEOC_BOVL_ml5 --sbovl_abs```

In [ ]:
![-h help] licsbas-13-sb-inv -h# EXAMPLES# licsbas-13-sb-inv -d GEOC_BOVL_ml5 -t TS_GEOC_BOVL_ml5 --sbovl# licsbas-13-sb-inv -d GEOC_BOVL_ml5 -t TS_GEOC_BOVL_ml5 --sbovl_abs --mem_size 8000

## Steps 1-4 → 1-6: **velocity STD / mask / filter (all `--sbovl`)**The same three steps as Module 1, but run with `--sbovl` (and, on the *absolute* chain, `--sbovl_abs`). For the full flag reference + the interactive viewers, see [**Module 1 →**](tutorial_1.ipynb).

In [ ]:
![-h help] licsbas-14-vel-std -h# EXAMPLE# licsbas-14-vel-std -t TS_GEOC_BOVL_ml5 --sbovl --mem_size 4000

In [ ]:
![-h help] licsbas-15-mask-ts -h# EXAMPLE (sbovl thresholds differ from the standard ones)# licsbas-15-mask-ts -t TS_GEOC_BOVL_ml5 --sbovl -c 0.05 -u 0.5 -g 50 -s 10 -i 1000

In [ ]:
![-h help] licsbas-16-filt-ts -h# EXAMPLE# licsbas-16-filt-ts -t TS_GEOC_BOVL_ml5 --sbovl -r 1

## Viewers & exportsThe interactive `plot-ts`/`out2nc`/`net2tif` viewer cells from [Module 1](tutorial_1.ipynb) work on a SBOVNL `TS_...` dir exactly as on a standard one.**Other routes:** [Module 0 — LiCSAR data](tutorial_0.ipynb) · [Module 0 — NISAR data](tutorial_0_nisar.ipynb) · [Module 1 — Time series (standard)](tutorial_1.ipynb)**[Continue in `tutorial_1.ipynb` (opens in this Binder session) &raquo;](<tutorial_1.ipynb>)**- [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/comet-licsar/LiCSBAS/HEAD?labpath=tutorial/tutorial_1.ipynb) - *new Binder session*- [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-licsar/LiCSBAS/blob/towards-v3/tutorial/tutorial_1.ipynb) - *Colab*